# 第七章：序列模型与 NLP 基础

## 7.1 为什么需要序列模型？

前几章的 MLP 和 CNN 处理的是**固定大小的输入**（如 28×28 图片）。但现实世界的数据很多是**序列**：

- 文本：单词序列 "I love deep learning"
- 语音：音频帧序列
- 时间序列：股票价格、天气数据
- 视频：帧序列

序列数据有三个核心挑战：
1. **可变长度**：不同句子长度不同（"Hi" vs "The quick brown fox..."）
2. **时序依赖**：后面的词依赖前面的词（"我喜欢吃"后面大概率是"苹果"而非"汽车"）
3. **长距离依赖**："我出生在巴黎……所以我法语讲得很好"——"巴黎"和"法语"相隔很远

### 为什么不直接用 MLP？

- MLP 需要**固定维度的输入**（将所有序列 pad 到相同长度 → 浪费）
- MLP **不共享参数**——"Hello"在句首和句尾被视为完全不同的特征
- MLP **无法建模时序关系**——输入被当作无序集合


## 7.2 循环神经网络 (RNN)

RNN 通过**隐藏状态 (hidden state)** 在时间步之间传递信息：

### 数学定义

在时间步 $t$，RNN 接收当前输入 $\mathbf{x}_t$ 和上一时刻的隐藏状态 $\mathbf{h}_{t-1}$，计算新的隐藏状态 $\mathbf{h}_t$：

$$

\mathbf{h}_t = \tanh(\mathbf{W}_{xh} \mathbf{x}_t + \mathbf{W}_{hh} \mathbf{h}_{t-1} + \mathbf{b}_h)

$$

输出（可选）：

$$

\mathbf{y}_t = \text{softmax}(\mathbf{W}_{hy} \mathbf{h}_t + \mathbf{b}_y)

$$

关键特性：**所有时间步共享相同的权重矩阵 $\mathbf{W}_{xh}, \mathbf{W}_{hh}, \mathbf{W}_{hy}$**——这就是为什么 RNN 可以处理任意长度的序列。

### BPTT (Backpropagation Through Time)

RNN 的反向传播沿着时间展开：
1. 前向：计算 $t=1,2,\ldots,T$ 的所有 $\mathbf{h}_t$ 和 $\mathbf{y}_t$
2. 反向：从 $t=T$ 开始，逐时间步应用链式法则

BPTT 的根本问题——**梯度消失/爆炸**：

$$

\frac{\partial \mathcal{L}}{\partial \mathbf{h}_1} = \frac{\partial \mathcal{L}}{\partial \mathbf{h}_T} \cdot \prod_{k=2}^{T} \frac{\partial \mathbf{h}_k}{\partial \mathbf{h}_{k-1}}

$$

如果 $|\frac{\partial \mathbf{h}_k}{\partial \mathbf{h}_{k-1}}| < 1$，连乘后梯度指数衰减 → RNN 无法学习长距离依赖。


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

# === RNN Cell 从零实现 ===
class RNNCell:
    """单步 RNN: h_t = tanh(W_xh @ x_t + W_hh @ h_{t-1} + b_h)"""
    def __init__(self, input_dim, hidden_dim):
        # Xavier 初始化
        self.W_xh = np.random.randn(hidden_dim, input_dim) * np.sqrt(2.0 / (input_dim + hidden_dim))  # 标准正态分布随机数
        self.W_hh = np.random.randn(hidden_dim, hidden_dim) * np.sqrt(2.0 / (hidden_dim + hidden_dim))  # 标准正态分布随机数
        self.b_h = np.zeros((hidden_dim, 1))  # 创建全零数组
    
    def forward(self, x_t, h_prev):
        """x_t: (input_dim, 1), h_prev: (hidden_dim, 1)"""
        z = self.W_xh @ x_t + self.W_hh @ h_prev + self.b_h
        return np.tanh(z)  # 双曲正切激活函数

# 演示：滚动序列
cell = RNNCell(input_dim=3, hidden_dim=5)
h = np.zeros((5, 1))  # 创建全零数组

# 输入序列：3 个 token，每个 token 3 维
sequence = [np.array([[1.0], [0.5], [0.0]]),  # 创建 NumPy 数组
            np.array([[0.0], [1.0], [0.5]]),  # 创建 NumPy 数组
            np.array([[0.5], [0.0], [1.0]])]  # 创建 NumPy 数组

print("RNN 前向传播:")
for t, x_t in enumerate(sequence):
    h = cell.forward(x_t, h)
    print(f"  t={t}: h = {h.T[0].round(3)}")
print(f"\n最终隐藏状态记录了前 3 步的信息 (input_dim=3, hidden_dim=5)")


## 7.3 LSTM：长短期记忆网络

LSTM (1997, Hochreiter & Schmidhuber) 通过精心设计的**门控机制**解决了 RNN 的梯度消失问题。

### LSTM 的三个门

| 门 | 公式 | 功能 |
|----|------|------|
| **遗忘门** $f_t$ | $\sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$ | 决定丢弃哪些旧记忆 |
| **输入门** $i_t$ | $\sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$ | 决定写入哪些新信息 |
| **输出门** $o_t$ | $\sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$ | 决定输出什么 |

### LSTM 核心：细胞状态 (Cell State)

与 RNN 只有隐藏状态不同，LSTM 有**两个状态**：

- **细胞状态 $c_t$**：长期记忆（高速公路，梯度可以不受阻碍地流动）
- **隐藏状态 $h_t$**：短期输出

更新规则：

$$

\tilde{c}_t = \tanh(W_c \cdot [h_{t-1}, x_t] + b_c)  \quad \text{(候选记忆)}

$$

$$

c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t  \quad \text{(选择性遗忘 + 选择性写入)}

$$

$$

h_t = o_t \odot \tanh(c_t)  \quad \text{(输出)}

$$

### 为什么 LSTM 解决梯度消失？

细胞状态 $c_t$ 的更新是**线性加法**（而非矩阵乘法！）：

$$

c_t = f_t c_{t-1} + i_t \tilde{c}_t

$$

梯度 $\frac{\partial c_t}{\partial c_{t-1}} = f_t$——只要 $f_t \approx 1$（遗忘门打开），梯度就不会衰减。


In [ ]:
# === LSTM Cell 从零实现 ===
import numpy as np
def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))  # 限制数值范围，防止溢出

class LSTMCell:
    def __init__(self, input_dim, hidden_dim):
        dim = input_dim + hidden_dim
        # 四组参数合并（计算效率）
        scale = np.sqrt(2.0 / dim)  # 平方根
        self.W_f = np.random.randn(hidden_dim, dim) * scale  # forget gate
        self.b_f = np.zeros((hidden_dim, 1)) + 1.0  # bias=1 鼓励初始时记住
        
        self.W_i = np.random.randn(hidden_dim, dim) * scale  # input gate
        self.b_i = np.zeros((hidden_dim, 1))  # 创建全零数组
        
        self.W_c = np.random.randn(hidden_dim, dim) * scale  # candidate
        self.b_c = np.zeros((hidden_dim, 1))  # 创建全零数组
        
        self.W_o = np.random.randn(hidden_dim, dim) * scale  # output gate
        self.b_o = np.zeros((hidden_dim, 1))  # 创建全零数组
    
    def forward(self, x_t, h_prev, c_prev):
        combined = np.vstack([h_prev, x_t])  # (hidden+input, 1)
        
        f_t = sigmoid(self.W_f @ combined + self.b_f)    # 遗忘门
        i_t = sigmoid(self.W_i @ combined + self.b_i)    # 输入门
        c_tilde = np.tanh(self.W_c @ combined + self.b_c) # 候选记忆
        o_t = sigmoid(self.W_o @ combined + self.b_o)    # 输出门
        
        c_t = f_t * c_prev + i_t * c_tilde   # 细胞状态更新
        h_t = o_t * np.tanh(c_t)             # 隐藏状态
        
        return h_t, c_t  # 返回结果

# 演示 LSTM
lstm_cell = LSTMCell(input_dim=3, hidden_dim=5)
h = np.zeros((5, 1))  # 创建全零数组
c = np.zeros((5, 1))  # 创建全零数组

print("LSTM 前向传播 (3 个时间步):")
for t, x_t in enumerate(sequence):
    h, c = lstm_cell.forward(x_t, h, c)
    print(f"  t={t}: h={h.T[0].round(3)}")
    print(f"       c={c.T[0].round(3)}")
    print(f"       c 的范数: {np.linalg.norm(c):.2f} (不会指数衰减)")  # 计算向量/矩阵范数


## 7.4 GRU：门控循环单元

GRU (2014, Cho et al.) 是 LSTM 的简化版——性能相近但参数少 25%，训练更快。

### GRU vs LSTM

| 特性 | LSTM | GRU |
|------|------|-----|
| 门数量 | 3 (遗忘/输入/输出) | 2 (重置/更新) |
| 状态数量 | 2 (c, h) | 1 (h) |
| 参数量 | 4× 组 | 3× 组 |
| 长距离依赖 | 最强 | 稍弱但足够 |

### GRU 更新规则

**重置门 $r_t$**（控制忽略多少过去信息）：

$$

r_t = \sigma(W_r \cdot [h_{t-1}, x_t] + b_r)

$$

**更新门 $z_t$**（控制保留多少过去信息 vs 新信息）：

$$

z_t = \sigma(W_z \cdot [h_{t-1}, x_t] + b_z)

$$

**候选隐藏状态**：

$$

\tilde{h}_t = \tanh(W \cdot [r_t \odot h_{t-1}, x_t] + b)

$$

**最终隐藏状态**（$z_t$ 作为插值系数）：

$$

h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t

$$

关键：当 $z_t \approx 0$，$h_t \approx h_{t-1}$——信息可以无损传递（类似 LSTM 的 carousel）。


## 7.5 NLP 基础：从文字到向量

### Tokenization（分词）

深度学习模型不能直接处理文字——需要将文字转换为数字序列。

**三种粒度：**

| 方法 | 示例 | 优点 | 缺点 |
|------|------|------|------|
| 词级 (Word) | "I love NLP" → [1, 42, 78] | 直观 | 词汇表巨大，OOV 问题 |
| 字符级 (Char) | "love" → [l, o, v, e] | 词汇表小 | 序列过长，语义弱 |
| 子词级 (Subword/BPE) | "unbelievable" → [un, believ, able] | **业界标准** | 实现较复杂 |

### Word Embeddings（词嵌入）

将离散的 token ID 映射为连续的稠密向量，使得语义相似的词在向量空间中距离更近：

$$

\mathbf{e}(\text{"king"}) - \mathbf{e}(\text{"man"}) + \mathbf{e}(\text{"woman"}) \approx \mathbf{e}(\text{"queen"})

$$

**Word2Vec (2013, Mikolov)** 开创了词嵌入时代：
- **Skip-gram**：用中心词预测上下文词
- **CBOW**：用上下文词预测中心词

**GloVe (2014, Pennington)** 基于全局词共现矩阵的因子分解。

现代 LLM 中，词嵌入是 Transformer 的第一层（可学习的 Embedding 层）。


In [ ]:
# === PyTorch: LSTM 字符级语言模型 ===
# 任务：用前几个字符预测下一个字符（最简单的语言建模）

# 准备数据
import torch
text = "hello world! this is a simple character level language model demo. " * 20
chars = sorted(list(set(text)))
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}
vocab_size = len(chars)

print(f"词汇表大小: {vocab_size}")
print(f"字符集: {''.join(chars)}")

# 创建训练序列
seq_length = 20
X_data, y_data = [], []
for i in range(0, len(text) - seq_length):
    seq_in = text[i:i + seq_length]
    seq_out = text[i + 1:i + seq_length + 1]
    X_data.append([char_to_idx[c] for c in seq_in])
    y_data.append([char_to_idx[c] for c in seq_out])

X_tensor = torch.tensor(X_data, dtype=torch.long)  # 从数据创建张量
y_tensor = torch.tensor(y_data, dtype=torch.long)  # 从数据创建张量
dataset = TensorDataset(X_tensor, y_tensor)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

print(f"训练样本: {len(X_data)}, 每批: 64")


In [ ]:
# === LSTM 字符级语言模型 ===
import torch
import torch.nn as nn
import torch.optim as optim
class CharLSTM(nn.Module):  # 所有网络层的基类
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)  # 词嵌入层
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers,  # 长短期记忆网络
                            batch_first=True, dropout=0.3)
        self.fc = nn.Linear(hidden_dim, vocab_size)  # 全连接层 y=Wx+b
    
    def forward(self, x, hidden=None):
        x = self.embedding(x)  # (batch, seq_len, embed_dim)
        out, hidden = self.lstm(x, hidden)
        out = self.fc(out)  # (batch, seq_len, vocab_size)
        return out, hidden  # 返回结果

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CharLSTM(vocab_size).to(device)  # 将数据移到 GPU 或 CPU
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.003)

print(f"模型参数: {sum(p.numel() for p in model.parameters()):,}")
print(model)


In [ ]:
# === 训练 ===
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
history = []
for epoch in range(15):
    model.train()  # 切换到训练模式（启用 Dropout/BatchNorm）
    total_loss = 0
    for x_batch, y_batch in loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)  # 将数据移到 GPU 或 CPU
        
        optimizer.zero_grad()  # 清空上一轮的梯度
        output, _ = model(x_batch)
        loss = criterion(output.reshape(-1, vocab_size), y_batch.reshape(-1))  # 改变张量形状
        loss.backward()  # 反向传播，计算梯度
        optimizer.step()  # 用累积的梯度更新参数
        total_loss += loss.item()  # 取出单个数值
    
    avg_loss = total_loss / len(loader)
    history.append(avg_loss)
    if epoch % 3 == 0:
        print(f"Epoch {epoch:2d}: loss = {avg_loss:.4f}")

# 画训练曲线
plt.figure(figsize=(8, 3))  # 创建画布
plt.plot(history, 'b-')  # 画线图
plt.xlabel('Epoch'); plt.ylabel('Loss')  # 设置x轴标签
plt.title('Char-LSTM Training')  # 设置图表标题
plt.grid(True, alpha=0.3)  # 显示网格线
plt.savefig('../fig/lstm_training.png', dpi=100, bbox_inches='tight')  # 保存图片到文件
plt.show()  # 显示图表

# === 文本生成 ===
@torch.no_grad()
def generate(model, seed_text, length=100, temperature=0.8):
    model.eval()  # 切换到评估模式（禁用 Dropout/BatchNorm）
    chars_gen = list(seed_text)
    hidden = None
    
    input_seq = torch.tensor([[char_to_idx[c] for c in seed_text]], device=device)  # 从数据创建张量
    
    for _ in range(length):
        output, hidden = model(input_seq, hidden)
        # 取最后一个时间步的预测，应用温度
        logits = output[0, -1, :] / temperature
        probs = F.softmax(logits, dim=-1)
        next_idx = torch.multinomial(probs, 1).item()  # 取出单个数值
        
        chars_gen.append(idx_to_char[next_idx])
        # 用新字符作为下一步输入
        input_seq = torch.tensor([[next_idx]], device=device)  # 从数据创建张量
    
    return ''.join(chars_gen)  # 返回结果

seed = "hello "
generated = generate(model, seed, length=80, temperature=0.7)
print(f"\n生成文本 (seed: '{seed}'):")
print(generated)


## 7.6 序列模型全景对比

| 模型 | 年份 | 核心机制 | 适用场景 | 状态 |
|------|------|---------|---------|------|
| **RNN** | 1986 | 简单循环连接 | 教学、简单序列 | 仅教学用 |
| **LSTM** | 1997 | 门控 + 细胞状态 | 长序列、预测 | 仍在广泛使用 |
| **GRU** | 2014 | 简化门控 | 类似 LSTM 但更轻量 | 仍在广泛使用 |
| **BiLSTM** | - | 双向 LSTM | 序列标注、NER | 工业界常见 |
| **Seq2Seq + Attention** | 2014 | Encoder-Decoder + 注意力 | 机器翻译 | 被 Transformer 取代 |
| **Transformer** | 2017 | 纯注意力、无循环 | **几乎所有 NLP** | 绝对主流 |

### 何时选择 RNN/LSTM 而非 Transformer？

- **数据量少**（< 10K 样本）：LSTM 收敛更快
- **需要流式处理**（逐 token 处理，不需要完整序列）
- **时间序列预测**：LSTM 的归纳偏置（时间连续性）在此任务上仍然有效
- **资源受限**：小 LSTM 比小 Transformer 表现更好


## 7.7 NLP 任务全景

在进入 Transformer/LLM 章节之前，了解 NLP 的经典任务体系：

| 任务 | 输入 → 输出 | 示例 |
|------|-----------|------|
| **文本分类** | 文本 → 标签 | 情感分析、垃圾邮件检测 |
| **序列标注** | token序列 → 标签序列 | NER、词性标注 |
| **文本生成** | prompt → 续写 | 故事生成、代码补全 |
| **机器翻译** | 源语言 → 目标语言 | en→zh |
| **文本摘要** | 长文本 → 短文本 | 新闻摘要 |
| **问答 (QA)** | (问题, 上下文) → 答案 | SQuAD |
| **文本蕴含 (NLI)** | (前提, 假设) → {蕴含,矛盾,中性} | MNLI |

这些任务在 LLM 时代被统一为"文本到文本"的范式——但理解它们各自的难度和评估方法仍然重要。


## 本章小结

| 概念 | 核心公式/机制 |
|------|-------------|
| **RNN** | $h_t = \tanh(W_{xh}x_t + W_{hh}h_{t-1})$，参数共享 |
| **梯度消失** | $\prod \partial h_k/\partial h_{k-1}$ 连乘指数衰减 |
| **LSTM** | 遗忘门 + 输入门 + 输出门 → 线性细胞状态更新 |
| **GRU** | 重置门 + 更新门（LSTM 简化版） |
| **Tokenization** | 词级/字符级/子词级 (BPE) |
| **Word2Vec** | Skip-gram + CBOW → 语义空间的向量表示 |
| **语言模型** | 给定前文预测下一个 token: $P(x_t | x_{<t})$ |

✅ 掌握了 RNN/LSTM 和 NLP 基础，下一章进入 Transformer 和 LLM 的世界，你会发现一切都有迹可循。
